# Google Sheet downloader

In [ ]:
import pandas as pd
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
import io
import os
import pickle

credentials_file = "path/to/credentials.json"
token_file = "path/to/token.pickle"

# Authenticate and build the Google Drive service using OAuth 2.0
def authenticate_drive():
    SCOPES = ['https://www.googleapis.com/auth/drive']
    creds = None

    # Check if token.pickle already exists for automatic login
    if os.path.exists(token_file):
        with open(token_file, 'rb') as token:
            creds = pickle.load(token)

    # If there are no valid credentials available, prompt the user to log in
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(credentials, SCOPES)
            creds = flow.run_local_server(port=0)

        # Save the credentials for future use
        with open(token_file 'wb') as token:
            pickle.dump(creds, token)

    service = build('drive', 'v3', credentials=creds)
    return service

# Download the spreadsheet as CSV from a Shared Drive
def download_spreadsheet(service, file_id, team_drive_id, output_csv_path):
    request = service.files().export_media(
        fileId=file_id, 
        mimeType='text/csv',
        #supportsAllDrives=True
    )
    with io.FileIO(output_csv_path, 'wb') as file:
        file.write(request.execute())
    print(f"File saved as {output_csv_path}")

# Load CSV into Pandas DataFrame
def load_csv_to_dataframe(csv_path):
    df = pd.read_csv(csv_path)
    return df

# Example usage
if __name__ == '__main__':
    # Initialize service
    service = authenticate_drive()

    # Define your file_id, team_drive_id, and output path
    file_id = '<file ID>'             # Replace with your actual file ID
    team_drive_id = '<team_drive ID'  # Replace with your Team Drive ID
    output_csv_path = '<path/to/file.csv>

    # Download the file and save it as CSV
    download_spreadsheet(service, file_id, team_drive_id, output_csv_path)

    # Load the CSV into a DataFrame
    df = load_csv_to_dataframe(output_csv_path)
    print(df.head())  # Display the first few rows of the DataFrame
